In [ ]:
"""
Hungarian 알고리즘 기반 평가 코드 (Verbose: 초과 번역 / 번역 실패)
------------------------------------------------------------------
1) build_gt_dict:
   - TASK_PATH 내 .json(G.T) 파일들에서 Subtask 이름을 추출하여
     gt_dict[파일명] = [subtask이름1, ...] 형태로 저장
2) main:
   - RESULT_PATH 내 jcci_top1(등) 폴더 아래 '작업 폴더'(simple1_4subtasks...) 탐색
   - 각 작업 폴더 내부 '타임스탬프 폴더' → approach/dag_bayesian.json 파싱
   - Hungarian 매칭으로 번역 성공률, 실행 성공률 등 계산
   - 번역 성공률=1.0인 케이스만 별도 Makespan 평균
   - 매칭 과정에서 leftover가 발생하면 (초과 번역 / 번역 실패) 구간 Verbose 출력
"""

import json
from pathlib import Path
from typing import List, Tuple, Set

import numpy as np
from scipy.optimize import linear_sum_assignment
from sentence_transformers import SentenceTransformer, util


from src.utils.constants import TASK_PATH, RESULT_PATH

model = SentenceTransformer("all-MiniLM-L6-v2")

--- Verbose Info ---
File: /home/dongkyu/pdk_ws/research/assets/results/jcci_top1/complex9_17subtasks(dc2, dnc1, dnc2, dnc3, nd(1, 2, 3)).json/2025-03-20_03_47_03_cook egg fry an.json/approach/dag_bayesian.json
  [초과 번역] ['Navigate to StoveKnob|-00.02|+00.88|-02.19 during 2.4000000000000004', 'Monitoring for Turn off Stove after cooking_12d97283', 'Wait for Turn off Stove after cooking', 'Wash Plate_part_1', 'Wash Plate_part_3']
  [번역 실패] ['Start Microwave', 'Wash Lettuce', 'Wash Apple', 'Wash Tomato']
  [실행 성공+초과 번역] ['Navigate to StoveKnob|-00.02|+00.88|-02.19 during 2.4000000000000004', 'Monitoring for Turn off Stove after cooking_12d97283', 'Wait for Turn off Stove after cooking', 'Wash Plate_part_1', 'Wash Plate_part_3']
  [실행 성공+번역 실패] ['Start Microwave', 'Wash Lettuce', 'Wash Apple', 'Wash Tomato']

--- Verbose Info ---
File: /home/dongkyu/pdk_ws/research/assets/results/jcci_top1/complex9_17subtasks(dc2, dnc1, dnc2, dnc3, nd(1, 2, 3)).json/2025-03-20_03_37_51_cook egg fry an.jso

In [ ]:
###############################################################################
# 1) Ground Truth 파일 → gt_dict
###############################################################################
def build_gt_dict() -> dict:
    """
    TASK_PATH 아래 .json 파일들을 순회:
    파일명 -> 해당 JSON 내 "Subtasks" 키 아래 "Name" 필드의 목록
    """
    gt_dict = {}
    for file in TASK_PATH.iterdir():
        if file.is_file() and file.suffix == ".json":
            file_name = file.name
            sub_names = []
            with file.open("r", encoding="utf-8") as f:
                data = json.load(f)
            for item in data:
                if "Subtasks" in item:
                    for subtask in item["Subtasks"]:
                        if "Name" in subtask:
                            sub_names.append(subtask["Name"])
            gt_dict[file_name] = sub_names
    return gt_dict

In [ ]:
###############################################################################
# 2) 헝가리안 알고리즘으로 1:1 매칭 + leftover(초과 번역/번역 실패) 계산
###############################################################################
def hungarian_match(
    recognized_names: List[str], gt_names: List[str], threshold: float
) -> Tuple[int, Set[int], Set[int]]:
    """
    recognized_names vs gt_names → Sentence-BERT 임베딩
    cos_sim 계산 후 "비용행렬 = (1 - sim)"로 변환
    linear_sum_assignment(헝가리안 알고리즘)으로 1:1 최대 매칭
    그 중 cos_sim >= threshold인 (i,j) 쌍만 "실제 매칭"으로 본다.

    반환:
      match_count: 실제 매칭된 쌍의 개수
      leftover_rec: 매칭되지 않은 '인지 Subtask' 인덱스 집합
      leftover_gt:  매칭되지 않은 'G.T Subtask' 인덱스 집합
    """
    R = len(recognized_names)
    G = len(gt_names)
    if R == 0 or G == 0:
        # 둘 중 하나라도 비어있으면, 매칭 자체가 없음
        return 0, set(range(R)), set(range(G))

    rec_embs = model.encode(recognized_names, convert_to_tensor=True)
    gt_embs = model.encode(gt_names, convert_to_tensor=True)
    sim_matrix = util.cos_sim(rec_embs, gt_embs).cpu().numpy()  # shape (R, G)

    cost_matrix = 1.0 - sim_matrix
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    matched_pairs = []
    for i, j in zip(row_ind, col_ind):
        if sim_matrix[i, j] >= threshold:
            matched_pairs.append((i, j))

    match_count = len(matched_pairs)
    matched_rec = {p[0] for p in matched_pairs}
    matched_gt = {p[1] for p in matched_pairs}

    leftover_rec = set(range(R)) - matched_rec  # 인지되었으나 매칭X
    leftover_gt = set(range(G)) - matched_gt  # G.T에 있으나 매칭X

    return match_count, leftover_rec, leftover_gt

In [ ]:
###############################################################################
# 3) dag_bayesian.json 1개 파일 → 헝가리안 매칭 지표 + leftover 반환
###############################################################################
def get_info_hungarian(
    dag_data: dict, gt_subtask_names: List[str], threshold: float
) -> Tuple[float, float, float, float, List[str], List[str], List[str], List[str]]:
    """
    1) Makespan
    2) subtask_success_rate = (실행 성공 매칭 수) / (인지 매칭 수)
    3) translation_success  = (인지 매칭 수) / (G.T Subtask 수)
    4) final_success        = translation_success × subtask_success_rate

    + leftover 목록(초과 번역, 번역 실패)
      - leftover_rec: 인지 Subtask 중 매칭 실패
      - leftover_gt : G.T 중 매칭 실패
      - leftover_rec_succ: "실행 성공 Subtask" 중 매칭 실패
      - leftover_gt_succ : G.T 중 매칭 실패(실행 성공과 매칭 기대)
    """
    subtask_exec_infos = dag_data["plans"][0]["subtasks"]
    makespan = subtask_exec_infos[-1]["endTime"] if subtask_exec_infos else 0.0

    recognized_names = [s["subtaskName"] for s in subtask_exec_infos]
    success_names = [
        s["subtaskName"] for s in subtask_exec_infos if s["executionStatus"]
    ]
    gt_count = len(gt_subtask_names)

    # (A) 번역(인지) 성공 매칭
    match_count, leftover_rec_inds, leftover_gt_inds = hungarian_match(
        recognized_names, gt_subtask_names, threshold
    )

    # (B) 실행 성공 매칭
    success_match_count, leftover_rec_succ_inds, leftover_gt_succ_inds = (
        hungarian_match(success_names, gt_subtask_names, threshold)
    )

    translation_success = match_count / gt_count if gt_count > 0 else 0.0
    subtask_success_rate = (
        (success_match_count / match_count) if match_count > 0 else 0.0
    )
    final_success = translation_success * subtask_success_rate

    # leftover 목록을 실제 "이름"으로 변환
    leftover_rec = [recognized_names[i] for i in sorted(leftover_rec_inds)]
    leftover_gt = [gt_subtask_names[i] for i in sorted(leftover_gt_inds)]
    leftover_rec_succ = [success_names[i] for i in sorted(leftover_rec_succ_inds)]
    leftover_gt_succ = [gt_subtask_names[i] for i in sorted(leftover_gt_succ_inds)]

    return (
        makespan,
        subtask_success_rate,
        translation_success,
        final_success,
        leftover_rec,
        leftover_gt,
        leftover_rec_succ,
        leftover_gt_succ,
    )

In [ ]:
###############################################################################
# 4) 특정 task_folder 내부: 타임스탬프 폴더들 순회
###############################################################################
def process_time_folders(
    task_folder: Path, gt_subtask_names: List[str], threshold: float
) -> Tuple[float, float, float, float, int, float, int]:
    """
    - task_folder 내부 타임스탬프 폴더 → approach/dag_bayesian.json
    - get_info_hungarian 로 각 파일을 처리
    - leftover(초과 번역, 번역 실패)가 있으면 파일명 함께 출력
    - 번역 100% 케이스만 별도 Makespan 누적
    """
    total_makespan = 0.0
    total_sub_succ = 0.0
    total_trans_succ = 0.0
    total_final_succ = 0.0
    file_count = 0

    perfect_trans_makespan_sum = 0.0
    perfect_trans_count = 0

    for timestamp_folder in task_folder.iterdir():
        if not timestamp_folder.is_dir():
            continue

        dag_file = timestamp_folder / "approach" / "dag_bayesian.json"
        if not dag_file.exists():
            continue

        with dag_file.open("r", encoding="utf-8") as f:
            dag_data = json.load(f)

        (
            makespan,
            sub_succ_rate,
            trans_succ_rate,
            final_succ,
            leftover_rec,
            leftover_gt,
            leftover_rec_succ,
            leftover_gt_succ,
        ) = get_info_hungarian(dag_data, gt_subtask_names, threshold)

        file_count += 1
        total_makespan += makespan
        total_sub_succ += sub_succ_rate
        total_trans_succ += trans_succ_rate
        total_final_succ += final_succ

        # 번역 성공률=1.0 → 모든 G.T와 매칭됨
        if abs(trans_succ_rate - 1.0) >= 0:
            perfect_trans_makespan_sum += makespan
            perfect_trans_count += 1

        # leftover 있으면 Verbose 출력
        # "초과 번역" = leftover_rec  (인지만 됐고 매칭 실패)
        # "번역 실패" = leftover_gt   (G.T 남음)
        # (실행 성공 leftover도 별도 표시 가능)
        if leftover_rec or leftover_gt:
            print(f"--- Verbose Info ---")
            print(f"File: {dag_file}")
            if leftover_rec:
                print(f"  [초과 번역] {leftover_rec}")
            if leftover_gt:
                print(f"  [번역 실패] {leftover_gt}")
            # 아래 2줄은 옵션 (실행 성공 leftover)
            if leftover_rec_succ:
                print(f"  [실행 성공+초과 번역] {leftover_rec_succ}")
            if leftover_gt_succ:
                print(f"  [실행 성공+번역 실패] {leftover_gt_succ}")
            print("")

    return (
        total_makespan,
        total_sub_succ,
        total_trans_succ,
        total_final_succ,
        file_count,
        perfect_trans_makespan_sum,
        perfect_trans_count,
    )

In [ ]:
###############################################################################
# 5) 상위 폴더 (ex: jcci_top1) → task_folder 순회
###############################################################################
def main(folder_name: str, gt_dict: dict, threshold: float):
    target_folder = RESULT_PATH / folder_name
    if not target_folder.exists():
        print(f"[ERROR] 폴더가 존재하지 않습니다: {target_folder}")
        return

    grand_makespan = 0.0
    grand_sub_succ = 0.0
    grand_trans_succ = 0.0
    grand_final_succ = 0.0
    grand_file_count = 0

    grand_perfect_makespan = 0.0
    grand_perfect_count = 0

    for item in target_folder.iterdir():
        # 예: simple1_4subtasks(dc1, nd1).json
        if item.is_dir() and item.name in gt_dict:
            gt_sub_names = gt_dict[item.name]

            (
                sum_makespan,
                sum_sub_succ,
                sum_trans_succ,
                sum_final_succ,
                cnt_files,
                sum_perfect_makespan,
                cnt_perfect,
            ) = process_time_folders(item, gt_sub_names, threshold)

            grand_makespan += sum_makespan
            grand_sub_succ += sum_sub_succ
            grand_trans_succ += sum_trans_succ
            grand_final_succ += sum_final_succ
            grand_file_count += cnt_files

            grand_perfect_makespan += sum_perfect_makespan
            grand_perfect_count += cnt_perfect

    # 통계 출력
    print("=========================================================")
    if grand_file_count > 0:
        avg_makespan = grand_makespan / grand_file_count
        avg_sub_succ = grand_sub_succ / grand_file_count
        avg_trans_succ = grand_trans_succ / grand_file_count
        avg_final_succ = grand_final_succ / grand_file_count

        print(
            f"[{folder_name}] (threshold={threshold:.2f}) 총 {grand_file_count}개 dag_bayesian.json"
        )
        print(f"  - Makespan 합/평균 : {grand_makespan:.3f} / {avg_makespan:.3f}")
        print(f"  - SubtaskSuccess   : {avg_sub_succ:.3f} (인지 매칭 중 실행 성공)")
        print(f"  - TranslationSucc  : {avg_trans_succ:.3f} (Hungarian, G.T 대비 인지)")
        print(f"  - FinalSuccess     : {avg_final_succ:.3f} (번역×실행)")

        if grand_perfect_count > 0:
            perfect_avg_makespan = grand_perfect_makespan / grand_perfect_count
            print()
            print(f"  * 번역 성공률=1.0 케이스: {grand_perfect_count}")
            print(f"  * 이 케이스들의 Makespan 평균: {perfect_avg_makespan:.3f}")
        else:
            print()
            print("  * 번역 성공률=1.0 케이스 없음.")
    else:
        print(f"[{folder_name}] - 분석할 파일이 없습니다.")

In [ ]:
###############################################################################
# 6) 실행부
###############################################################################
if __name__ == "__main__":
    # G.T 파일 로드
    gt_dict = build_gt_dict()

    # 평가 대상 폴더
    folder_names = ["jcci_top1", "jcci_top25", "jcci_zeroshot"]

    # 유사도 임계값 (조정 가능)
    threshold = 0.8

    for folder_name in folder_names:
        main(folder_name, gt_dict, threshold)